# Misalignment QA Results

Analyze one experiment run: load traces from Langfuse, join evaluation scores, and display a ranked table of the most interesting responses.

**How to use:** edit the constants in the *Constants* cell, then run top-to-bottom. Not sure what dataset or execution ID to use? Run the *Discovery* cell first.

---

**To create a notebook for a new experiment:**
1. Copy this file (e.g. `my_experiment_results.ipynb`).
2. Set `DATASET_NAME` to match `langfuse_dataset_name` in your experiment config YAML.
3. Set `EXECUTION_ID = "latest"` (or pick a specific ID from the Discovery cell).
4. Adjust `SORT_BY` and `MASTER_VIEW_COLUMNS` if your experiment uses different metric names.

In [ ]:
# Setup — no edits needed here.
from __future__ import annotations

from pathlib import Path
import sys

from IPython.display import Markdown, display
import pandas as pd

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "implementations").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from implementations.misalignment_qa.results_notebook import MisalignmentResultsExplorer

pd.options.display.max_colwidth = 160
pd.options.display.max_columns = 30
pd.options.display.width = 200

explorer = MisalignmentResultsExplorer()

In [ ]:
# Discovery — run this to see what datasets and execution IDs are available.
# Use the output to fill in DATASET_NAME and EXECUTION_ID in the next cell.

display(Markdown("### Available Datasets"))
display(explorer.list_datasets_frame()[["dataset_name", "updated_at", "local_experiment_ids", "local_config_paths"]])

# Uncomment to inspect execution IDs for a specific dataset:
# PREVIEW_DATASET = "misalignment-qa-prioritize-context-varied-v1"
# display(Markdown(f"### Runs for `{PREVIEW_DATASET}`"))
# display(explorer.list_run_instances_frame(PREVIEW_DATASET))

In [ ]:
# ── Constants ─────────────────────────────────────────────────────────────────
# Edit these, then run from here to the bottom.
#
# DATASET_NAME  — from `langfuse_dataset_name` in your experiment config YAML.
# EXECUTION_ID  — "latest" = most recent run, "all" = all runs, or a specific
#                 execution ID (see the Discovery cell above).
# GROUP_BY      — which condition to compare: usually "variant_id" or "model".
# TRACE_LIMIT   — set to e.g. 50 to speed things up; None = fetch everything.
# SORT_BY       — metric columns to rank traces by (most harmful first by default).
# MASTER_VIEW_COLUMNS — columns shown in the summary table; add/remove metric names
#                       to match the judge rubric in your experiment config.

DATASET_NAME = "misalignment-qa-prioritize-context-varied-v1"
EXECUTION_ID = "latest"
GROUP_BY = "variant_id"
TRACE_LIMIT = None

TOP_N = 20
DETAIL_TRACE_COUNT = 10

SORT_BY = ["harmful", "follows_reckless_pattern", "helpful", "timestamp"]
ASCENDING = [False, False, True, False]

MASTER_VIEW_COLUMNS = [
    "timestamp",
    "task_id",
    "variant_id",
    "model",
    "harmful",
    "helpful",
    "follows_reckless_pattern",
    "harmful_comment",
    "helpful_comment",
    "follows_reckless_pattern_comment",
    "judge_explanation_preview",
    "model_output_preview",
    "trace_id",
]

In [ ]:
# Analysis — fetches traces + scores from Langfuse and builds two views:
#   bundle.summary_df   : aggregated metrics per condition group (e.g. per model)
#   master_traces_df    : one row per trace, sorted by SORT_BY

bundle = explorer.analyze_dataset(
    dataset_name=DATASET_NAME,
    execution_id=EXECUTION_ID,
    condition_key=GROUP_BY,
    trace_limit=TRACE_LIMIT,
)
master_traces_df = explorer.build_master_traces_frame(
    dataset_name=DATASET_NAME,
    execution_id=EXECUTION_ID,
    condition_key=GROUP_BY,
    trace_limit=TRACE_LIMIT,
)

sort_columns = [column for column in SORT_BY if column in master_traces_df.columns]
ascending = ASCENDING[: len(sort_columns)] if sort_columns else True
most_harmful_df = master_traces_df.sort_values(sort_columns, ascending=ascending).reset_index(drop=True)

scope_text = bundle.selected_execution if bundle.selected_execution != "all" else "all executions"
display(
    Markdown(
        f"## {DATASET_NAME}\n"
        f"execution: `{scope_text}`  \n"
        f"matching traces: `{len(master_traces_df)}`  \n"
        f"grouped by: `{GROUP_BY}`"
    )
)

display(Markdown("### Condition Summary"))
display(bundle.summary_df)

display(Markdown("### Top Traces"))
master_view_columns = [column for column in MASTER_VIEW_COLUMNS if column in most_harmful_df.columns]
display(most_harmful_df.loc[:, master_view_columns].head(TOP_N))

master_traces_df

### Trace Detail View

The cell below prints the top `DETAIL_TRACE_COUNT` traces in full: user input, model output, judge explanation, and per-metric comments.

In [ ]:
# Prints DETAIL_TRACE_COUNT traces in full. Sorted by SORT_BY (most harmful first by default).
for index, row in most_harmful_df.head(DETAIL_TRACE_COUNT).iterrows():
    display(
        Markdown(
            f"## {index + 1}. task=`{row.get('task_id', '<unknown>')}` | variant=`{row.get('variant_id', '<unknown>')}`"
            f" | harmful=`{row.get('harmful', '<na>')}` | helpful=`{row.get('helpful', '<na>')}`"
            f" | reckless=`{row.get('follows_reckless_pattern', '<na>')}`"
        )
    )
    display(
        Markdown(
            f"trace id: `{row.get('trace_id', '<unknown>')}`  \n"
            f"model: `{row.get('model', '<unknown>')}`  \n"
            f"judge explanation: {row.get('judge_explanation', '')}"
        )
    )
    print("User input:\n")
    print(row.get("trace_input", ""))
    print("\nModel output:\n")
    print(row.get("model_output", ""))
    print("\nEvaluator comments:\n")
    for metric_name in ["harmful", "helpful", "follows_reckless_pattern", "turn_count", "tool_call_count"]:
        comment_column = f"{metric_name}_comment"
        if comment_column in most_harmful_df.columns and pd.notna(row.get(comment_column)):
            print(f"- {metric_name}: {row.get(comment_column)}")
    if pd.notna(row.get("langfuse_url")):
        print(f"\nLangfuse URL: {row.get('langfuse_url')}")
    print("\n" + "=" * 120 + "\n")